In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# upload PR-CLIP.zip lên Drive trước, sửa đường dẫn cho đúng
!rm -rf /content/PR-CLIP
!unzip -q -o /content/drive/MyDrive/PR-CLIP.zip -d /content
%cd /content/PR-CLIP

Mounted at /content/drive
/content/PR-CLIP


In [ ]:
!pip install -q open_clip_torch==2.20.0 clip-benchmark==1.6.1 transformers==4.45.2 adapters timm sentence-transformers wandb pyyaml gdown
!pip install -q "protobuf>=6.31.1,<7"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.21.0 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-dataproc 5.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
grain 0.2.18 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
googleapis-common-protos 1.76.0 

In [ ]:
!gdown --fuzzy "https://drive.google.com/file/d/15htMe2iE4kVujjWURxfCxZXk8vAAetoT/view" -O images.zip
!rm -rf images && mkdir images
!tar -xzf images.zip -C images
!find images -type f -exec mv -t images {} + 2>/dev/null
!find images -mindepth 1 -type d -empty -delete
!rm images.zip
!mkdir -p cache/weights

Downloading...
From (original): https://drive.google.com/uc?id=15htMe2iE4kVujjWURxfCxZXk8vAAetoT
From (redirected): https://drive.google.com/uc?id=15htMe2iE4kVujjWURxfCxZXk8vAAetoT&confirm=t&uuid=0590365d-6052-4d8c-b354-286004c7438f
To: /content/PR-CLIP/images.zip
100% 1.46G/1.46G [00:23<00:00, 61.5MB/s]


In [ ]:
!mkdir -p /content/PR-CLIP/cache/weights
!wget -O /content/PR-CLIP/cache/weights/RET-2.pt \
  https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt
!ls -la /content/PR-CLIP/cache/weights/RET-2.pt   # khoảng 605MB

--2026-11-04 07:46:10--  https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/ckpt/RS5M_ViT-B-32_RET-2.pt
Resolving huggingface.co (huggingface.co)... 3.171.171.104, 3.171.171.128, 3.171.171.65, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.104|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/655fb2aadc7511e22748819d/48ec26e685f2b7622075158c0649039d64cdcbdbe150faedb6e79ea1d13d0e78?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27RS5M_ViT-B-32_RET-2.pt%3B+filename%3D%22RS5M_ViT-B-32_RET-2.pt%22%3B&user_id=public&xip=339JibobBhk&Expires=1793781970&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU1ZmIyYWFkYzc1MTFlMjI3NDg4MTlkLzQ4ZWMyNmU2ODVmMmI3NjIyMDc1MTU4YzA2NDkwMzlkNjRjZGNiZGJlMTUwZmFlZGI2ZTc5ZWExZDEzZDBlNzhcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMmeGlwPTMzOUppYm9iQmhrIiwiQ

In [ ]:
!sed -i '450s/default=False/default=True/' training/params.py
!grep -n "load-clip-weights\"" training/params.py   # phải thấy default=True

450:        "--load-clip-weights", default=True, action="store_true",


In [ ]:
import pandas as pd, os
for f in ['dataset/train/ret2.csv','dataset/test/rsicd_test.csv','dataset/test/rsitmd_test.csv']:
    names = pd.read_csv(f)['filename'].unique()
    missing = [n for n in names if not os.path.exists(os.path.join('images', n))]
    print(f, '| tổng:', len(names), '| thiếu:', len(missing))

dataset/train/ret2.csv | tổng: 13092 | thiếu: 0
dataset/test/rsicd_test.csv | tổng: 1093 | thiếu: 0
dataset/test/rsitmd_test.csv | tổng: 452 | thiếu: 0


In [ ]:
%cd /content/PR-CLIP
!sed -i 's/from timm.optim.nadam import Nadam/from timm.optim.nadam import NAdamLegacy as Nadam/' training/optimization.py
!sed -i 's/from timm.optim.radam import RAdam$/from timm.optim.radam import RAdamLegacy as RAdam/' training/optimization.py
!sed -n 5,12p training/optimization.py
!nvidia-smi | head -12
!sed -i 's/np\.string_/np.bytes_/g' data/train_data.py utils/tsne_checkpoints.py
!sed -i 's/astype(np\.float)/astype(float)/g' evaluation/linear_eval.py
!grep -n "np.bytes_" data/train_data.py

/content/PR-CLIP
from timm.optim.adafactor import Adafactor
from timm.optim.adahessian import Adahessian
from timm.optim.adamp import AdamP
from timm.optim.lookahead import Lookahead
from timm.optim.nadam import NAdamLegacy as Nadam
from timm.optim.radam import RAdamLegacy as RAdam
from timm.optim.rmsprop_tf import RMSpropTF
from timm.optim.sgdp import SGDP
Wed Nov  4 07:46:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+=============

In [ ]:
!pip install -q cn_clip --no-deps
!nvidia-smi
!pip install -q faiss-cpu click

Wed Nov  4 07:46:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%cd /content/PR-CLIP
import re
LOG_PCT = 10          # đổi thành 50 nếu chỉ muốn hiện 50% và 100%

s = open('training/train.py', encoding='utf-8').read().replace('\r\n', '\n')

old = "(i % 10 == 0 or batch_count == num_batches_per_epoch)"
new = "(batch_count in {round(num_batches_per_epoch * k * LOG_PCT / 100) for k in range(1, 100 // LOG_PCT + 1)})"
s = s.replace(old, new)

if 'LOG_PCT = ' not in s:
    anchor = "import torch.nn.functional as F\n"
    s = s.replace(anchor, anchor + f"\nLOG_PCT = {LOG_PCT}\n", 1)
else:
    s = re.sub(r"LOG_PCT = \d+", f"LOG_PCT = {LOG_PCT}", s, count=1)

open('training/train.py', 'w', encoding='utf-8').write(s)
!grep -n "LOG_PCT" training/train.py

/content/PR-CLIP
13:LOG_PCT = 10
205:            if is_master(args) and (batch_count in {round(num_batches_per_epoch * k * LOG_PCT / 100) for k in range(1, 100 // LOG_PCT + 1)}):


In [ ]:
%cd /content/PR-CLIP
s = open('training/train.py', encoding='utf-8').read().replace('\r\n', '\n')
old = "                scaler.unscale_(optimizer)\n                norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=args.max_grad_norm)\n"
new = old + '''                if i in (0, 5, 20, 40):
                    _bad = [n for n, p in model.named_parameters() if p.grad is not None and not torch.isfinite(p.grad).all()]
                    print(f"[DBG] step {i} loss={total_loss.item():.4f} scale={scaler.get_scale()} norm={norm} bad_params={len(_bad)}", _bad[:12], flush=True)
                if i >= 40:
                    raise SystemExit("[DBG] done")
'''
if '[DBG]' not in s:
    s = s.replace(old, new)
    open('training/train.py', 'w', encoding='utf-8').write(s)
print('[DBG]' in s)

/content/PR-CLIP
True


In [ ]:
%cd /content/PR-CLIP
import re

s = open('training/train.py', encoding='utf-8').read().replace('\r\n', '\n')
s = re.sub(
    r"                if i in \(0, 5, 20, 40\):\n.*?\n.*?\n                if i >= 40:\n                    raise SystemExit\(\"\[DBG-fp32\] done\"\)\n",
    "",
    s, flags=re.DOTALL, count=1
)
open('training/train.py', 'w', encoding='utf-8').write(s)
print('[DBG-fp32]' in s)

/content/PR-CLIP
False


In [ ]:
%cd /content/PR-CLIP
import re

def read(p): return open(p, encoding='utf-8').read().replace('\r\n', '\n')

NEW = '''            if scaler is not None:  # AMP-FIX
                scaler.scale(total_loss).backward()
                if (i + 1) % step_lou == 0 or i == train_loader_size - 1:
                    scaler.unscale_(optimizer)
                    norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=args.max_grad_norm)
                    if args.horovod:
                        optimizer.synchronize()
                        with optimizer.skip_synchronize():
                            scaler.step(optimizer)
                    else:
                        scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
'''
pat = re.compile(r"            if scaler is not None:\n                scaler\.scale\(total_loss\)\.backward\(\)\n.*?                scaler\.update\(\)\n", re.S)

s = read('training/train.py')
if '# AMP-FIX' not in s:
    s = pat.sub(lambda m: NEW, s, count=1)
    a = "    train_loader_size = len(dataloader)\n    step_lou = 4\n"
    s = s.replace(a, a + "    norm = torch.tensor(0.0)\n")
    open('training/train.py', 'w', encoding='utf-8').write(s)

!grep -c "AMP-FIX" training/train.py
!grep -c "\[DBG" training/train.py

/content/PR-CLIP
1
0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/PR-CLIP
import re

def read(p): return open(p, encoding='utf-8').read().replace('\r\n', '\n')
def write(p, s): open(p, 'w', encoding='utf-8').write(s)
def sub(path, pairs):
    s = read(path)
    for old, new in pairs:
        s = s.replace(old, new)
    write(path, s)


# 2. timm / numpy bản mới đổi tên + tắt log dài
sub('training/optimization.py', [
    ('from timm.optim.nadam import Nadam\n', 'from timm.optim.nadam import NAdamLegacy as Nadam\n'),
    ('from timm.optim.radam import RAdam\n', 'from timm.optim.radam import RAdamLegacy as RAdam\n'),
    ('logging.info("Param groups = %s" % json.dumps(parameter_group_names, indent=2))', 'pass'),
])
sub('data/train_data.py', [('np.string_', 'np.bytes_')])
sub('utils/tsne_checkpoints.py', [('np.string_', 'np.bytes_')])
sub('evaluation/linear_eval.py', [('astype(np.float)', 'astype(float)')])
sub('training/main.py', [
    ("logging.info(f'\\t{n}\\t{list(p.size())}')", 'pass'),
    ("logging.info('Model structure\\n' +str(model))", 'pass'),
    ('logging.info(f"  {name}: {val}")', 'pass'),
])

# 3. Thay return_hidden của open_clip bản sửa
helper = '''    def _vit_hidden(self, images):
        v = self.image_backbone
        x = v.conv1(images.type(v.conv1.weight.dtype))
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        cls = v.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
        x = torch.cat([cls, x], dim=1)
        x = x + v.positional_embedding.to(x.dtype)
        x = v.patch_dropout(x)
        x = v.ln_pre(x)
        x = x.permute(1, 0, 2)
        x = v.transformer(x)
        x = x.permute(1, 0, 2)
        return v.ln_post(x)

'''
s = read('model/model.py')
if '_vit_hidden' not in s:
    s = s.replace("self.image_backbone(images, return_hidden=True)", "self._vit_hidden(images)")
    marker = "    def forward(self, images, texts, text_only=False):"
    s = s.replace(marker, helper + marker)
    write('model/model.py', s)

# 4. In log theo % epoch (10 -> 10%,20%,...; 50 -> 50%,100%)
LOG_PCT = 10
s = read('training/train.py')
s = s.replace("(i % 10 == 0 or batch_count == num_batches_per_epoch)",
              "(batch_count in {round(num_batches_per_epoch * k * LOG_PCT / 100) for k in range(1, 100 // LOG_PCT + 1)})")
if 'LOG_PCT = ' not in s:
    anchor = "import torch.nn.functional as F\n"
    s = s.replace(anchor, anchor + f"\nLOG_PCT = {LOG_PCT}\n", 1)
else:
    s = re.sub(r"LOG_PCT = \d+", f"LOG_PCT = {LOG_PCT}", s, count=1)
write('training/train.py', s)


# Kiểm tra: phải ra 2, 1, 2, 2, True
!grep -c "_vit_hidden" model/model.py
!grep -c "NAdamLegacy" training/optimization.py
!grep -c "np.bytes_" data/train_data.py
!grep -c "LOG_PCT" training/train.py
print('[DBG-fp32]' in read('training/train.py'))


%cd /content/PR-CLIP
import re, py_compile
p = 'training/main.py'
s = open(p, encoding='utf-8').read().replace('\r\n', '\n')

if '[LITE-CKPT]' in s:
    print('Đã vá từ trước, không làm gì thêm.')
else:
    def sub_regex(pattern, repl, text, name):
        n = len(re.findall(pattern, text))
        assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
        return re.sub(pattern, repl, text, count=1)

    def sub_literal(old, new, text, name):
        n = text.count(old)
        assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
        return text.replace(old, new)

    # 1. Không lưu epoch_N.pt (chỉ còn best.pt + epoch_latest.pt)
    s = sub_literal('if completed_epoch == args.epochs or (', 'if False or (', s, 'epoch_N')

    # 2. epoch_latest.pt: ĐẦY ĐỦ (fp32 + optimizer + scaler), ghi qua file .tmp rồi đổi tên
    pat = r'( +)if args\.save_most_recent:\n\s+torch\.save\(\n\s+checkpoint_dict,\n\s+os\.path\.join\(args\.checkpoint_path, f"epoch_latest\.pt"\),\n\s+\)[ ]*\n'
    def rep_latest(m):
        i = m.group(1)
        return (f'{i}if args.save_most_recent:  # [LITE-CKPT]\n'
                f'{i}    _tmp = os.path.join(args.checkpoint_path, "epoch_latest.tmp")\n'
                f'{i}    torch.save(checkpoint_dict, _tmp)\n'
                f'{i}    os.replace(_tmp, os.path.join(args.checkpoint_path, "epoch_latest.pt"))\n')
    s = sub_regex(pat, rep_latest, s, 'epoch_latest')

    # 3. best.pt: NHẸ (fp16, không optimizer)
    pat = r'torch\.save\(\n(\s+)checkpoint_dict,\n(\s+)os\.path\.join\(args\.checkpoint_path, f"best\.pt"\),'
    def rep_best(m):
        return ('torch.save(\n' + m.group(1) +
                '{**checkpoint_dict, "optimizer": None, "state_dict": {k: (v.half() if v.is_floating_point() else v) for k, v in checkpoint_dict["state_dict"].items()}},\n' +
                m.group(2) + 'os.path.join(args.checkpoint_path, f"best.pt"),')
    s = sub_regex(pat, rep_best, s, 'best')

    # 4. Resume được cả khi checkpoint không có optimizer (vd best.pt nhẹ)
    s = sub_literal('                    if optimizer is not None:\n                        optimizer.load_state_dict(checkpoint["optimizer"])',
                    '                    if optimizer is not None and checkpoint.get("optimizer") is not None:\n                        optimizer.load_state_dict(checkpoint["optimizer"])',
                    s, 'resume')

    # 5. Train xong đủ epoch -> tự xóa epoch_latest (giữ best.pt)
    old = '    if args.wandb and is_master(args):\n        wandb.finish()\n'
    new = ('    if is_master(args) and args.save_logs:  # [AUTO-DEL]\n'
           '        for _f in ("epoch_latest.pt", "epoch_latest.tmp"):\n'
           '            _p = os.path.join(args.checkpoint_path, _f)\n'
           '            if os.path.exists(_p):\n'
           '                os.remove(_p)\n'
           '                logging.info(f"Train xong, da xoa {_p}")\n') + old
    s = sub_literal(old, new, s, 'auto_delete')

    open(p, 'w', encoding='utf-8').write(s)
    py_compile.compile(p, doraise=True)   # báo lỗi cú pháp ngay nếu vá sai
    print('Đã vá xong:', '[LITE-CKPT]' in s, '[AUTO-DEL]' in s)
    print('best.pt nhẹ (fp16, không optimizer) | epoch_latest.pt đầy đủ | tự xóa epoch_latest khi xong')

# ---- Vá thêm: resume trên PyTorch >= 2.6 + dừng ngay nếu thiếu checkpoint ----
import py_compile
p = 'training/main.py'
s = open(p, encoding='utf-8').read().replace('\r\n', '\n')

def _sub_once(old, new, text, name):
    n = text.count(old)
    assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
    return text.replace(old, new)

if 'weights_only=False' not in s:
    s = _sub_once('torch.load(args.resume, map_location=device)',
                  'torch.load(args.resume, map_location=device, weights_only=False)',
                  s, 'weights_only')
if 'Khong tim thay checkpoint' not in s:
    s = _sub_once('            logging.info("=> no checkpoint found at \'{}\'".format(args.resume))',
                  '            raise FileNotFoundError(f"Khong tim thay checkpoint: {args.resume}")',
                  s, 'resume_missing')
open(p, 'w', encoding='utf-8').write(s)
py_compile.compile(p, doraise=True)
print('Vá resume OK:', 'weights_only=False' in s, 'Khong tim thay checkpoint' in s)

!grep -n "load-clip-weights\"" training/params.py   # phải thấy default=True

%cd /content/PR-CLIP
p = 'model/model.py'
s = open(p, encoding='utf-8').read().replace('\r\n', '\n')

if 'CrossModalExtraction' in s:
    print('Đã vá từ trước, không làm gì thêm.')
else:
    def rep(old, new, text, name):
        n = text.count(old)
        assert n == 1, f'[{name}] khớp {n} chỗ (cần đúng 1)'
        return text.replace(old, new)

    CLS = '''class CrossModalExtraction(nn.Module):
    """Extraction (paper Eq.4-6): pre-LN + GELU, modality-type embedding, text padding mask."""
    def __init__(self, dim=512, heads=8, ff_dim=1024, layers=2, dropout=0.1, final_norm=False):
        super().__init__()
        self.type_embed = nn.Parameter(torch.zeros(2, 1, dim))
        nn.init.trunc_normal_(self.type_embed, std=0.02)
        layer = nn.TransformerEncoderLayer(dim, heads, ff_dim, dropout, activation='gelu',
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers, norm=nn.LayerNorm(dim) if final_norm else None,
                                             enable_nested_tensor=False)

    def forward(self, image_embed, text_embed, text_mask=None):
        B, Li, _ = image_embed.shape
        x = torch.cat([image_embed + self.type_embed[0], text_embed + self.type_embed[1]], dim=1)
        kpm = None
        if text_mask is not None:  # True = token thật
            kpm = torch.cat([torch.zeros(B, Li, dtype=torch.bool, device=x.device), ~text_mask.bool()], dim=1)
        out = self.encoder(x, src_key_padding_mask=kpm)
        return out[:, :Li], out[:, Li:]


'''
    # 1. thêm class
    s = rep('class FusionModel(nn.Module):', CLS + 'class FusionModel(nn.Module):', s, 'class')

    # 2. thay it_encoder bằng extraction
    s = rep('''        it_encoder_layer = nn.TransformerEncoderLayer(dim_str, 8, 1024, 0.1, batch_first=True)
        self.it_encoder = nn.TransformerEncoder(it_encoder_layer, 2)
''', '''        self.extraction = CrossModalExtraction(dim_str, 8, 1024, 2, 0.1)
''', s, 'build')

    # 3. lưu mask text thật (EOT) khi encode_text
    s = rep('''                texts = self.tokenizer(texts, context_length=context_length).to(self.device)
''', '''                texts = self.tokenizer(texts, context_length=context_length).to(self.device)
                self.text_valid_mask = (torch.arange(texts.size(1), device=texts.device)[None, :]
                                        <= texts.argmax(dim=-1)[:, None]) if self.prompt is None else None
''', s, 'mask')
    s = rep('        self.text_hidden = torch.tensor(1)\n',
            '        self.text_hidden = torch.tensor(1)\n        self.text_valid_mask = None\n', s, 'init')

    # 4. forward
    s = rep('        att_embed = self.it_encoder(it_seq)\n',
            '''        att_img, att_txt = self.extraction(image_embed, text_embed, self.text_valid_mask)
        att_embed = torch.cat([att_img, att_txt], dim=1)
''', s, 'forward')

    open(p, 'w', encoding='utf-8').write(s)
    import py_compile; py_compile.compile(p, doraise=True)
    print('Vá extraction xong, compile OK')

!grep -c "CrossModalExtraction" model/model.py   # phải ra 2
!grep -c "it_encoder" model/model.py             # phải ra 0

# Chạy thử fp32 (dừng sau ~40 bước)
!torchrun --nproc_per_node 1 -m training.main \
  --train-data ./dataset/train/ret2.csv --images-dir ./images \
  --retrieval-data ./dataset/test/rsicd_test.csv --retrieval-images-dir ./images \
  --csv-separator , --csv-img-key filename \
  --retrieval-csv-separator , --retrieval-csv-img-key filename \
  --retrieval-frequency 1 --save-frequency 0 \
  --epochs 1 --batch-size 16 --workers 2 --precision amp \
  --lr 2e-7 --warmup 100 --weight_decay 0.5 --max-grad-norm 50 \
  --image-model ViT-B-32 --image-model-builder openclip \
  --text-model ViT-B-32 --text-model-builder openclip \
  --pretrained-image-model --pretrained-text-model --loss InfoNCE \
  --seed 1 \
  --name rsicd_improve_extraction --logs /content/drive/MyDrive/prclip_logs

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/PR-CLIP
2
1
2
2
False
/content/PR-CLIP
Đã vá xong: True True
best.pt nhẹ (fp16, không optimizer) | epoch_latest.pt đầy đủ | tự xóa epoch_latest khi xong
Vá resume OK: True True
450:        "--load-clip-weights", default=True, action="store_true",
/content/PR-CLIP
Vá extraction xong, compile OK
2
0
I0000 00:00:1793778401.830821    2225 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793778404.052519    2225 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecate

In [ ]:
import os, torch
path = '/content/drive/MyDrive/prclip_logs/rsicd_bs16_seed3/checkpoint/epoch_latest.pt'
print('tồn tại:', os.path.exists(path))
ck = torch.load(path, map_location='cpu', weights_only=False)
print('epoch:', ck['epoch'], '| có optimizer:', ck['optimizer'] is not None, '| có scaler:', 'scaler' in ck)
del ck

tồn tại: False


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/prclip_logs/rsicd_bs16_seed3/checkpoint/epoch_latest.pt'

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())
!nvidia-smi | head -15
!ls -la /content/drive/MyDrive/prclip_logs/rsicd_train-10_18-11_06_31/checkpoints/